# Volatility surfaces (strike × expiry grids)

Reference for **two-dimensional implied volatility** over expiry and strike (or moneyness): smiles and skews.

## Concept

A **vol surface** attaches a Black–Scholes / Black **implied volatility** to each **(expiry, strike)** pair on a grid. **Smile** is how vol changes with strike at fixed expiry; **term structure** is how ATM vol changes with expiry. Surfaces are usually stored as **regular grids** with vols in **row-major** order for fast lookup and scenario bumps.

## API walkthrough

Try importing a surface type from the Python `market_data` API. If it is not bound yet, this notebook stays **conceptual** and prints what a grid workflow would look like.

In [ ]:
try:
    from finstack.core.market_data import VolSurface  # type: ignore[attr-defined]
except ImportError:
    print(
        "VolSurface is not exposed on finstack.core.market_data in this build.\n"
        "In Rust/core, surfaces are often stored with expiries[], strikes[], and "
        "vols_row_major[i_strike + i_expiry * n_strikes].",
    )
else:
    print("VolSurface import succeeded — extend this cell with construction once API is stable.")
    print(VolSurface)

## Practical example

**Synthetic grid (pure Python):** build a tiny **strike × expiry** vol matrix and **lookup** by nearest indices — mirrors how bindings expose row-major flat arrays.

In [ ]:
import math

expiries = [0.25, 0.5, 1.0]
strikes = [90.0, 100.0, 110.0]
# Stylized smile: higher vol away from 100; slight upward term structure
vols = []
for t in expiries:
    for k in strikes:
        m = abs(math.log(k / 100.0))
        atm = 0.20 + 0.02 * math.sqrt(t)
        smile = 0.04 * m
        vols.append(atm + smile)
print("expiries (years):", expiries)
print("strikes:", strikes)
print("flat vols row-major (n_exp * n_strike):", [round(v, 4) for v in vols])


def vol_at(exp_idx: int, strike_idx: int) -> float:
    n_k = len(strikes)
    return vols[strike_idx + exp_idx * n_k]


print("ATM 1Y vol (K=100, T=1.0):", round(vol_at(2, 1), 4))
print("Interpretation: wing strikes show smile uplift vs ATM at each expiry.")

## Takeaways

- **2D grids** couple **expiries** and **strikes**; **row-major** packing is the common serialization.
- **Smile / skew** = slice at **fixed T**; **term structure** = slice at **fixed K** (often ATM).
- When **`VolSurface`** lands in Python, replace the toy grid with **`MarketContext`** wiring and instrument **vol_surface_id** hooks.